# The Greening (and Browning) of the Albuquerque Metro Area — replication

This notebook reproduces the "quadrant map" showing where water use across the Albuquerque, NM
metro area has been **rising** or **falling** over the last two decades. It classifies every
30-meter pixel in the Albuquerque Census Urban Area into one of five categories by combining:

- **Baseline** — how much water the land surface used (evapotranspiration net of precipitation)
  in 2001-2005, split into **wet** vs. **dry** halves relative to the rest of the metro area, and
- **Trend** — whether that water use has been **gaining** or **losing** (or barely changing) from
  2001 to 2024.

The five resulting classes — Wet/gaining, Dry/gaining, Little change, Wet/losing, Dry/losing —
are a compact way of asking "where is the metro area's water footprint expanding, and where is it
shrinking?" without needing parcel-level data.

**Published figure** (`docs/quadrant_map/openet_quadrant_map_urbanarea_minus_river.png`) — the
version presented in the accompanying talk. This notebook reproduces it from public data sources
with one documented simplification (see "Notes and caveats" at the end): the published version
additionally subtracts a custom-digitized river-corridor shapefile that is not publicly
redistributable; this notebook uses the full Census Urban Area boundary instead, so results differ
slightly along the river channel itself.

## Method summary

- **Water-use metric:** *effective ET* = OpenET ensemble evapotranspiration − 0.7 × GRIDMET
  precipitation, both from Google Earth Engine, at 30 m resolution.
- **Baseline:** mean annual effective ET, 2001-2005.
- **Trend:** per-pixel linear regression slope of annual effective ET, 2001-2024 (units: mm/yr²).
- **Smoothing:** NaN-aware Gaussian filter (σ = 2 pixels, ≈ 60 m) applied to both the baseline and
  trend rasters before classifying, so the map reads as legible patches rather than
  salt-and-pepper noise.
- **Classification:** baseline **median split** (wet vs. dry, relative to this AOI) × trend
  **sign** (gaining vs. losing), with the bottom **tercile of |trend|** pulled out as a separate
  "Little Change" class. Thresholds are re-derived from this AOI's own distribution — they are
  *relative*, not absolute, so they should not be compared across differently-shaped AOIs.


## Setup — Google Earth Engine account

This notebook pulls satellite-derived data directly from **Google Earth Engine (GEE)**, a free
cloud platform for planetary-scale geospatial analysis. You'll need your own (free) GEE account
and a Google Cloud project with the Earth Engine API enabled — takes about five minutes:

1. **Sign up for Earth Engine** (no cost for research/non-commercial use):
   <https://earthengine.google.com/signup/>
2. **Create a Google Cloud project** (or reuse one you have) and enable the Earth Engine API:
   <https://console.cloud.google.com/> → *APIs & Services* → enable "Google Earth Engine API".
   Or register the project directly at <https://code.earthengine.google.com/register>.
3. **Install the Python client** if you haven't already: `pip install earthengine-api`.
4. **Replace the placeholder below** (`GEE_PROJECT`) with your own Google Cloud project ID.
5. Run the next cell. The first time, `ee.Authenticate()` opens a browser window for you to log
   in and grant access — after that, credentials are cached locally and you won't need to
   authenticate again on this machine.

**Requirements:** `earthengine-api`, `geopandas`, `requests`, `rasterio`, `scipy`, `numpy`,
`matplotlib` (Python 3.10+).


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import ee

# --- EDIT THIS: your own Google Cloud project ID with the Earth Engine API enabled ---
GEE_PROJECT = "your-gee-project-id"  # <-- replace with your project ID

ee.Authenticate()  # first run only: opens a browser login; cached afterward
ee.Initialize(project=GEE_PROJECT)
print("Earth Engine initialized with project:", GEE_PROJECT)


## Step 1 — Define the Albuquerque metro AOI (Census Urban Area, public data)

The area of interest is the Census Bureau's **"Albuquerque, NM Urban Area"** (2020 Urban Area
boundary, GEOID `01171`) — a single contiguous polygon covering the built-up Albuquerque metro
footprint (ABCWUA + Rio Rancho + Corrales + surrounding development), fetched live from the
Census Bureau's public **TIGERweb** REST API. No download or account needed for this step.

The published talk figure additionally subtracts a narrow river-corridor strip (the
levee-to-levee channel) from this polygon, using a custom-digitized shapefile that isn't publicly
distributed — see the caveats at the end. This notebook uses the **full Urban Area polygon**
without that subtraction, which is a public-data-only simplification.


In [ ]:
import geopandas as gpd
import requests
import shapely.geometry

TIGERWEB_URL = (
    "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/Urban/MapServer/8/query"
)

def fetch_urban_area(geoid="01171", name_contains="Albuquerque"):
    """Fetch a Census Urban Area polygon from TIGERweb by GEOID, falling back to a name search
    if the GEOID field name has changed vintage-to-vintage (TIGERweb has used both GEOID and
    GEOID20 across releases)."""
    field_attempts = [f"GEOID20='{geoid}'", f"GEOID='{geoid}'", f"UACE20='{geoid}'"]
    for where in field_attempts:
        resp = requests.get(TIGERWEB_URL, params={
            "where": where, "outFields": "*", "returnGeometry": "true", "f": "geojson",
        }, timeout=60)
        resp.raise_for_status()
        gj = resp.json()
        if gj.get("features"):
            gdf = gpd.GeoDataFrame.from_features(gj["features"], crs="EPSG:4326")
            print(f"Matched on '{where}' -> {len(gdf)} feature(s)")
            return gdf
    # Fallback: name search
    resp = requests.get(TIGERWEB_URL, params={
        "where": f"NAME LIKE '%{name_contains}%'", "outFields": "*",
        "returnGeometry": "true", "f": "geojson",
    }, timeout=60)
    resp.raise_for_status()
    gj = resp.json()
    if not gj.get("features"):
        raise RuntimeError(
            "Could not find the Albuquerque Urban Area via TIGERweb GEOID or name search. "
            "Check https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/Urban/MapServer/8 "
            "directly and adjust the query."
        )
    gdf = gpd.GeoDataFrame.from_features(gj["features"], crs="EPSG:4326")
    print(f"Matched by name search -> {len(gdf)} feature(s)")
    return gdf

aoi_gdf = fetch_urban_area()
label_field = next((c for c in ("NAME", "NAMELSAD20", "NAMELSAD") if c in aoi_gdf.columns), None)
if label_field:
    print("Urban Area:", aoi_gdf.iloc[0][label_field])

aoi_gdf_utm = aoi_gdf.to_crs("EPSG:32613")  # UTM 13N, matches the rest of this project's rasters
area_km2 = aoi_gdf_utm.area.sum() / 1e6
print(f"AOI area: {area_km2:,.1f} km2")

aoi_geom_4326 = shapely.geometry.mapping(aoi_gdf.dissolve().geometry.iloc[0])
aoi = ee.Geometry(aoi_geom_4326)


## Step 2 — Baseline effective ET (2001-2005)

*Effective ET* nets out the water that fell as rain from the water that left the land surface,
leaving a proxy for water use beyond what precipitation alone supplies:

`effective_ET = OpenET_ensemble − 0.7 × GRIDMET_precip`

Both terms come from public Earth Engine collections:

- **OpenET ensemble** (`OpenET/ENSEMBLE/CONUS/GRIDMET/MONTHLY/v2_0`) — monthly evapotranspiration
  (mm), already an ensemble of six ET models. Summed across the year gives annual ET.
- **GRIDMET** (`IDAHO_EPSCOR/GRIDMET`) — daily precipitation (mm), summed across the year gives
  annual precipitation.

The **baseline** is the mean of the annual effective-ET rasters for 2001, 2002, 2003, 2004, 2005.


In [ ]:
OPENET_COLLECTION = "OpenET/ENSEMBLE/CONUS/GRIDMET/MONTHLY/v2_0"
OPENET_BAND = "et_ensemble_mad"
GRIDMET_COLLECTION = "IDAHO_EPSCOR/GRIDMET"
GRIDMET_BAND = "pr"
PRECIP_COEF = 0.7  # open methodological choice — see "Notes and caveats"

def annual_effective_et(year):
    """Annual effective ET (mm) for a calendar year, clipped to the AOI."""
    start = ee.Date.fromYMD(year, 1, 1)
    end = ee.Date.fromYMD(year + 1, 1, 1)
    et = (ee.ImageCollection(OPENET_COLLECTION)
          .filterDate(start, end).select(OPENET_BAND)
          .sum().toDouble())
    precip = (ee.ImageCollection(GRIDMET_COLLECTION)
              .filterDate(start, end).select(GRIDMET_BAND)
              .sum().toDouble())
    eff_et = et.subtract(precip.multiply(PRECIP_COEF)).rename("eff_et")
    return eff_et.clip(aoi)

BASELINE_YEARS = list(range(2001, 2006))
baseline_stack = ee.ImageCollection([annual_effective_et(y) for y in BASELINE_YEARS])
baseline_image = baseline_stack.mean().rename("baseline")
print(f"Baseline: mean effective ET, {BASELINE_YEARS[0]}-{BASELINE_YEARS[-1]}")


## Step 3 — Per-pixel trend (2001-2024)

For each pixel, fit a straight line through its 24 annual effective-ET values
(2001 through 2024) and keep the **slope** — mm per year, per year (i.e. mm/yr²). A positive
slope means effective ET is rising at that pixel (more water being used than precipitation
supplies, and increasingly so); negative means it's falling.


In [ ]:
TREND_START, TREND_END = 2001, 2024

def annual_effective_et_ee(year_ee):
    """Same effective-ET calculation as Step 2, but built from an ee.Number so it can be used
    inside a server-side ee.List.map() over the trend years."""
    year = ee.Number(year_ee)
    start = ee.Date.fromYMD(year, 1, 1)
    end = ee.Date.fromYMD(year.add(1), 1, 1)
    et = (ee.ImageCollection(OPENET_COLLECTION)
          .filterDate(start, end).select(OPENET_BAND)
          .sum().toDouble())
    precip = (ee.ImageCollection(GRIDMET_COLLECTION)
              .filterDate(start, end).select(GRIDMET_BAND)
              .sum().toDouble())
    return et.subtract(precip.multiply(PRECIP_COEF)).rename("eff_et").clip(aoi)

def _prep(year):
    year = ee.Number(year)
    img = annual_effective_et_ee(year)
    return (img.addBands(ee.Image.constant(1).toFloat().rename("constant"))
               .addBands(ee.Image(year.subtract(TREND_START)).toFloat().rename("t"))
               .select(["constant", "t", "eff_et"]))

years = ee.List.sequence(TREND_START, TREND_END)
annual_collection = ee.ImageCollection(years.map(_prep))

fit = annual_collection.reduce(ee.Reducer.linearRegression(numX=2, numY=1))
trend_image = (fit.select("coefficients")
               .arrayProject([0]).arrayFlatten([["constant", "slope"]])
               .select("slope").rename("trend").clip(aoi))
print(f"Trend: per-pixel linear slope of effective ET, {TREND_START}-{TREND_END}")


## Step 4 — Download both rasters

Both rasters are requested as GeoTIFFs at 30 m resolution in UTM Zone 13N (EPSG:32613), matching
the rest of this project's spatial outputs. The Albuquerque Urban Area (~690 km²) at 30 m is
about 750,000 pixels per raster — comfortably inside Earth Engine's synchronous per-request
download limit, so no Google Drive export step is needed here.


In [ ]:
import urllib.request
from pathlib import Path

OUT_CRS = "EPSG:32613"
SCALE = 30

def find_package_root(start=None):
    cur = Path(start or Path.cwd()).resolve()
    for p in [cur, *cur.parents]:
        if (p / "data" / "quadrant_map").is_dir():
            return p
    raise FileNotFoundError(
        "Could not find the package root (containing data/quadrant_map) above the notebook.")

PKG_ROOT = find_package_root()
OUT_DIR = PKG_ROOT / "data" / "quadrant_map" / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def download_image(image, out_path, band_name):
    url = image.select(band_name).getDownloadURL({
        "scale": SCALE, "region": aoi, "crs": OUT_CRS,
        "format": "GEO_TIFF", "filePerBand": False,
    })
    urllib.request.urlretrieve(url, out_path)
    mb = out_path.stat().st_size / 1e6
    print(f"Wrote {out_path.name} ({mb:.1f} MB)")

baseline_path = OUT_DIR / "openet_effective_et_baseline_2001_2005.tif"
trend_path = OUT_DIR / "openet_effective_et_trend_2001_2024.tif"
download_image(baseline_image, baseline_path, "baseline")
download_image(trend_image, trend_path, "trend")


## Step 5 — Classify into five water-use categories

Both rasters are smoothed with a **NaN-aware Gaussian filter** (σ = 2 pixels, ≈ 60 m) before
classifying — smoothing `value × mask` and `mask` separately and dividing, so no-data edges don't
bleed incorrect values into the AOI interior. Then:

1. **Baseline** is split at its own **median** into "wet" (above median) vs. "dry" (below median).
2. **Trend** is split by **sign** into "gaining" (positive slope) vs. "losing" (negative slope).
3. Pixels whose **|trend|** falls in the **bottom third** (tercile) of the AOI's trend-magnitude
   distribution are reclassified as **"Little Change"**, overriding the wet/dry × gaining/losing
   split — these are pixels where nothing meaningful is happening either way.

This produces five classes: **Wet/gaining, Dry/gaining, Little Change, Wet/losing, Dry/losing.**
Because the thresholds (median, tercile) are computed from *this AOI's own* pixel distribution,
they are relative rankings, not absolute water-use levels — don't compare class assignments
across a different AOI without re-deriving thresholds there too.


In [ ]:
import numpy as np
import rasterio
from scipy.ndimage import gaussian_filter

SIGMA = 2  # ~60 m at 30 m/pixel

def read_band(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float64")
        nodata = src.nodata
        profile = src.profile.copy()
    if nodata is not None:
        arr[arr == nodata] = np.nan
    return arr, profile

def nan_aware_smooth(arr, sigma):
    """Gaussian-smooth an array with NaNs without letting NaNs contaminate valid pixels."""
    mask = np.isfinite(arr).astype(float)
    filled = np.nan_to_num(arr, nan=0.0)
    smoothed_values = gaussian_filter(filled, sigma=sigma)
    smoothed_mask = gaussian_filter(mask, sigma=sigma)
    with np.errstate(invalid="ignore", divide="ignore"):
        result = smoothed_values / smoothed_mask
    result[smoothed_mask < 1e-6] = np.nan
    return result

baseline_raw, profile = read_band(baseline_path)
trend_raw, _ = read_band(trend_path)

baseline_smooth = nan_aware_smooth(baseline_raw, SIGMA)
trend_smooth = nan_aware_smooth(trend_raw, SIGMA)

valid = np.isfinite(baseline_smooth) & np.isfinite(trend_smooth)
baseline_median = np.nanmedian(baseline_smooth[valid])
trend_abs = np.abs(trend_smooth[valid])
trend_tercile_cutoff = np.nanpercentile(trend_abs, 100 / 3)

print(f"Baseline median: {baseline_median:.2f} mm/yr")
print(f"Trend |slope| bottom-tercile cutoff: {trend_tercile_cutoff:.3f} mm/yr^2")

# Class codes: 1 Little Change, 2 Dry-gaining, 3 Dry-losing, 4 Wet-gaining, 5 Wet-losing
cls = np.zeros(baseline_smooth.shape, dtype="uint8")
wet = baseline_smooth >= baseline_median
gaining = trend_smooth >= 0
little_change = np.abs(trend_smooth) < trend_tercile_cutoff

cls[valid & little_change] = 1
cls[valid & ~little_change & ~wet & gaining] = 2
cls[valid & ~little_change & ~wet & ~gaining] = 3
cls[valid & ~little_change & wet & gaining] = 4
cls[valid & ~little_change & wet & ~gaining] = 5

for code_val, label in [(1, "Little Change"), (2, "Dry, gaining"), (3, "Dry, losing"),
                        (4, "Wet, gaining"), (5, "Wet, losing")]:
    n = int((cls == code_val).sum())
    pct = 100 * n / int(valid.sum())
    print(f"  {code_val} {label:<14}: {n:>10,} px  ({pct:5.1f}%)")


## Step 6 — Render the map


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimage
import matplotlib.font_manager as fm
from matplotlib.patches import Patch

SLATE = "#0f172a"
GREY = "#5a5a5a"
FONT = "Helvetica" if "Helvetica" in {f.name for f in fm.fontManager.ttflist} else "Arial"
plt.rcParams["font.family"] = FONT

CLASS_COLORS = {1: "#d9d9d9", 2: "#80cdc1", 3: "#dfc27d", 4: "#01665e", 5: "#8c510a"}
LEGEND = [(4, "Wet, gaining water"), (2, "Dry, gaining water"), (1, "Little change"),
          (5, "Wet, losing water"), (3, "Dry, losing water")]

with rasterio.open(baseline_path) as src:
    b = src.bounds
extent = [b.left, b.right, b.bottom, b.top]

bg = np.zeros((*cls.shape, 4), dtype=float)
for v, hexc in CLASS_COLORS.items():
    r, g, bl = int(hexc[1:3], 16) / 255, int(hexc[3:5], 16) / 255, int(hexc[5:7], 16) / 255
    bg[cls == v] = [r, g, bl, 1.0]
bg[cls == 0] = [0, 0, 0, 0]  # nodata -> transparent

fig, ax = plt.subplots(figsize=(11, 12.5))
fig.subplots_adjust(left=0.03, right=0.97, top=0.90, bottom=0.06)
ax.imshow(bg, extent=extent, origin="upper", interpolation="nearest")
ax.set_xlim(extent[0], extent[1])
ax.set_ylim(extent[2], extent[3])
ax.set_axis_off()

fig.text(0.5, 0.955, "The Greening (and Browning) of the Albuquerque Metro Area",
         ha="center", va="center", fontsize=20, fontweight="bold", color=SLATE)
fig.text(0.5, 0.918,
         "Albuquerque, NM Census Urban Area — effective ET (OpenET - 0.7xprecip), 2001-24\n"
         "(replication: full Urban Area boundary, no river-corridor subtraction)",
         ha="center", va="center", fontsize=11.5, color=GREY)

handles = [Patch(facecolor=CLASS_COLORS[v], edgecolor="none", label=lab) for v, lab in LEGEND]
leg = ax.legend(handles=handles, title="Water use, 2001-24", loc="upper left",
                fontsize=10.5, title_fontsize=11, framealpha=0.92, borderpad=0.8)
leg.get_frame().set_edgecolor("#cccccc")

LOGO = PKG_ROOT / "docs" / "logo.png"
if LOGO.exists():
    logo = mpimage.imread(LOGO)
    logo_h = 0.036
    logo_w = logo_h * (logo.shape[1] / logo.shape[0]) * (12.5 / 11)
    lax = fig.add_axes([0.03, 0.02, logo_w, logo_h], anchor="SW", zorder=10)
    lax.imshow(logo); lax.axis("off")
fig.text(0.97, 0.025, "Data: OpenET, GRIDMET, U.S. Census Bureau",
         ha="right", va="bottom", fontsize=8.5, color=GREY)

out_png = OUT_DIR / "quadrant_map_replication.png"
fig.savefig(out_png, dpi=200)
print(f"Wrote {out_png}")
plt.show()


## Compare to the published figure

The published talk figure (built with the additional river-corridor subtraction described below)
is included in this repository at `docs/quadrant_map/openet_quadrant_map_urbanarea_minus_river.png`
for direct comparison.


## Notes and caveats

- **AOI simplification.** The published figure subtracts a narrow river-corridor strip (the
  levee-to-levee channel) from the Urban Area polygon before classifying, using a shapefile
  custom-digitized by a research collaborator that is not publicly redistributable. This notebook
  uses the **full Urban Area polygon** without that subtraction — expect visible differences
  along the river channel itself (open water and riverbed have their own ET signal, which is noise
  for a residential-land-use story), but the metro-wide pattern should closely match.
- **The 0.7x precipitation coefficient is a modeling choice, not a physical constant** — it
  approximates the fraction of precipitation available to plants/soil before runoff and
  evaporation losses, but has not been locally calibrated. A different coefficient would shift the
  effective-ET baseline and, likely to a lesser extent, the trend.
- **Relative classification, not absolute levels.** Median and tercile thresholds are re-derived
  from this specific AOI's pixel distribution. The same pixel could be classified differently if
  run over a smaller or larger AOI — don't compare class assignments across different geographies.
- **OpenET resolution (~30 m) and modeling uncertainty.** OpenET is itself an ensemble of six
  remote-sensing ET models; it is well-validated at agricultural field scale but carries more
  uncertainty over heterogeneous urban land cover. Treat this as a neighborhood-scale pattern, not
  a parcel-level or lot-level measurement.
- **A fresh OpenET ensemble pull may differ slightly from the published run** — OpenET has
  updated its ensemble methodology and underlying model versions over time (this replication uses
  the `v2_0` ensemble collection, current as of the published run's vintage).
